## Imports

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, roc_curve, roc_auc_score

## Data Loading & Preprocessing

In [ ]:
sp500 = yf.Ticker("^GSPC")
sp500 = sp500.history(period="max")

In [ ]:
del sp500["Dividends"]
del sp500["Stock Splits"]

In [ ]:
sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [ ]:
sp500 = sp500.loc["1990-01-01":].copy()

## Feature Engineering

In [ ]:
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()

    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]

    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]

    new_predictors += [ratio_column, trend_column]

In [ ]:
sp500 = sp500.dropna()
sp500.shape

## Model Training & Backtesting

In [ ]:
model = LogisticRegression(random_state=1, max_iter=1000)

In [ ]:
def predict(train, test, predictors, model):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train[predictors])
    test_scaled = scaler.transform(test[predictors])

    model.fit(train_scaled, train["Target"])

    probs = model.predict_proba(test_scaled)[:, 1]

    preds = (probs >= .6).astype(int)

    result = pd.DataFrame({
        "Target": test["Target"].values,
        "Predictions": preds,
        "Probabilities": probs
    }, index=test.index)

    return result

In [ ]:
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)

    return pd.concat(all_predictions)

In [ ]:
predictions = backtest(sp500, model, new_predictors)
predictions["Predictions"].value_counts()

## Evaluation & Visualization

In [ ]:
precision = precision_score(predictions["Target"], predictions["Predictions"])
print(f"Precision: {precision:.4f}")

recall = recall_score(predictions["Target"], predictions["Predictions"])
print(f"Recall: {recall:.4f}")

In [ ]:
auc = roc_auc_score(predictions["Target"], predictions["Probabilities"])
print(f"AUC: {auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(predictions["Target"], predictions["Probabilities"])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve; S&P 500 Prediction')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()